# Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual <a href='https://quantra.quantinsti.com/quantra-notebook' target="_blank">here</a>.
1. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
1. Before running this notebook on your local PC:<br>
i.  You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on "**Run Codes Locally on Your Machine**" in the course.<br>
ii. You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or Python modules that might be required to run this notebook.

# Managing Trading Positions with KiteConnect

Effective portfolio management is just as important as order execution. You need a clear, real-time view of your open positions to track profitability, manage risk, and make informed decisions. The KiteConnect API provides a straightforward way to access all your positions, whether they are for the current day or held overnight.

This notebook will guide you through:

1.  **Fetching Positions**: How to retrieve your current day and net positions.
2.  **Analyzing Position Data**: Understanding key metrics like Profit & Loss (P&L), quantity, and average price.
3.  **Converting Positions**: How to programmatically convert an intraday position to an overnight one and vice-versa.

## Prerequisites

* **Open Positions**: To see meaningful data, you should have some open positions in your account when running this notebook during market hours.

## Authentication

We begin by authenticating our session to get a live `kite` object.

In [ ]:
import pandas as pd
from kiteconnect import KiteConnect

# A helper class to manage Kite Connect authentication.
class KiteAuth:
    def __init__(self, api_key, api_secret):
        self.api_key, self.api_secret, self.kite = api_key, api_secret, None
    
    # Method to authenticate and generate a session.
    def authenticate(self):
        try:
            kite = KiteConnect(api_key=self.api_key)
            # Generate a login URL to get the request_token.
            print("Login URL: ", kite.login_url())
            request_token = input("Enter request_token: ").strip()
            
            # Generate a session using the request_token.
            data = kite.generate_session(request_token, api_secret=self.api_secret)
            
            # Set the access_token for future API calls.
            kite.set_access_token(data['access_token'])
            self.kite = kite
            print("Authentication Successful!")
            return kite
        except Exception as e:
            print(f"Auth Error: {e}")
            return None

# --- Start Authentication ---
# IMPORTANT: Replace with your own API Key and Secret.
API_KEY = "YOUR_API_KEY"
API_SECRET = "YOUR_API_SECRET"

kite = None
if API_KEY != "YOUR_API_KEY":
    auth = KiteAuth(API_KEY, API_SECRET)
    kite = auth.authenticate()
else:
    print("Please update your API_KEY and API_SECRET to run.")

## Fetching Positions

This step demonstrates how to retrieve your current trading positions. The `kite.positions()` method provides two types of positions:
1. Net: This is for the actual, current net position of your portfolio. 
2. Day: This is for positions taken during the day.

We'll display these positions, including relevant details like the trading symbol, quantity, average price, last traded price, and profit/loss.

In [2]:
# Check if the KiteConnect object is authenticated
if kite:
    try:
        # Fetch all current trading positions
        positions = kite.positions()

        # Net positions
        net_positions = positions['net']
        if net_positions:
            print("--- Net Positions ---")
            # Convert the list of dictionaries to a DataFrame
            net_df = pd.DataFrame(net_positions)
            # Display selected columns for clarity
            display(net_df[['tradingsymbol', 'quantity', 'average_price', 'last_price', 'pnl']])
        else:
            print("No overnight positions.")

        # Day positions
        day_positions = positions['day']
        if day_positions:
            print("\n--- Day Positions ---")
            # Convert the list of dictionaries to a pandas DataFrame
            day_df = pd.DataFrame(day_positions)
            # Display selected columns
            display(day_df[['tradingsymbol', 'quantity', 'average_price', 'last_price', 'm2m']])
        else:
            print("\nNo intraday positions.")

    except Exception as e:
        # Handle any errors during the fetching of positions
        print(f"Error fetching positions: {e}")

--- Net Positions ---


,tradingsymbol,quantity,average_price,last_price,pnl
0,INFY,1,1605.9,1606.8,0.9



--- Day Positions ---


,tradingsymbol,quantity,average_price,last_price,m2m
0,INFY,1,1605.9,1606.8,0.9


## Analyzing Position Data

The positions DataFrame gives you all the information you need to assess your portfolio's performance.
- **`quantity`**: Positive for long positions, negative for short positions.
- **`average_price`**: The weighted average price at which you entered the position.
- **`pnl`**: The total Profit and Loss for the position.
- **`m2m`**: Mark-to-Market P&L, which is the unrealized P&L for the current day.

Let's calculate the total Mark-to-Market P&L for all our intraday trades. The M2M P&L represents the unrealized profit or loss for positions held within the current trading day.

In [3]:
if 'day_df' in locals() and not day_df.empty:
    # Calculate the sum of the 'm2m' column
    total_day_pnl = day_df['m2m'].sum()
    print(f"Total Day Mark-to-Market P&L: {total_day_pnl:.2f}")
else:
    print("No Day positions to analyze.")

Total Day Mark-to-Market P&L: 0.90


## Converting Positions

Sometimes you might want to convert an intraday position (MIS) to an overnight one (CNC/NRML) to carry it forward, or vice-versa. The `kite.convert_position()` method allows you to do this programmatically.

**Important:** Position conversion is subject to margin availability. If you don't have enough margin to hold an MIS position overnight as CNC/NRML, the request will be rejected.

### Example: Convert an MIS Buy Position to CNC

Let's assume we have an open long position in 'INFY' under the `MIS` product type and we want to convert it to `CNC` for delivery.

In [4]:
# This is a demonstrative example. It requires an actual open MIS position in INFY.
if kite and 'day_df' in locals():
    # Find the INFY MIS position
    infy_mis_position = day_df[(day_df['tradingsymbol'] == 'INFY') & (day_df['product'] == 'MIS')]

    if not infy_mis_position.empty:
        position = infy_mis_position.iloc[0]

        # Ensure it's a long position before converting
        if position['quantity'] > 0:
            try:
                print(f"Attempting to convert {position['quantity']} shares of INFY from MIS to CNC...")

                # Call the convert_position method
                response = kite.convert_position(
                    exchange=position['exchange'],
                    tradingsymbol=position['tradingsymbol'],
                    transaction_type=kite.TRANSACTION_TYPE_BUY, # Must match the original trade
                    position_type='day', # The source position type
                    quantity=position['quantity'],
                    old_product=kite.PRODUCT_MIS,
                    new_product=kite.PRODUCT_CNC
                )

                # Check the response to confirm successful conversion
                if response:
                    print("SUCCESS: Position conversion successful.")
                else:
                    print("FAILED: Position conversion failed. Check response or margins.")

            except Exception as e:
                # Handle any exceptions that occur during the conversion process
                print(f"FAILED to convert position: {e}")
        else:
            print("Cannot convert a short MIS position to CNC.")
    else:
        print("No open MIS position found for INFY to convert.")
else:
    print("Kite not authenticated or no intraday positions available.")

No open MIS position found for INFY to convert.


Note: To exit a position, you can place a counter order.

## Conclusion

Monitoring and managing your positions is a continuous activity for any trader. You are now able to programmatically fetch all your open positions, analyze their profitability, and execute conversions between product types.

This functionality is the final piece of the puzzle, enabling you to build a complete, end-to-end trading application that not only places orders but also manages the resulting portfolio according to your strategy's rules.
<br><br>